# CO2 Solubility in Aqueous Amines — Kent-Eisenberg Screening Model

This notebook demonstrates NeqSim's screening-level model for the solubility of
carbon dioxide in aqueous alkanolamine solvents (MEA, DEA, MDEA, activated MDEA).
It implements the classic **Kent-Eisenberg** approach: apparent (lumped)
equilibrium constants describe acid-gas speciation, and physically dissolved free
CO2 is related to its partial pressure through Henry's law:

$$ p_{CO_2} = H_{CO_2}(T)\,[CO_2]_{\text{free}} $$

**What this notebook shows**
1. The validated screening API (static model and the `AmineSystem` wrapper).
2. Reproduction of the regression-test anchors (MDEA partial pressure, molarity).
3. CO2 partial pressure vs loading for MEA and MDEA at 40 °C — primary vs tertiary binding.
4. Temperature effect (40 °C absorber vs 100 °C stripper) — the basis of solvent regeneration.
5. Differential heat of absorption by amine type.

The model targets **process screening** (solvent selection, loading and circulation
estimates). In the engineering loading windows it reproduces literature isotherms
to within roughly a factor of two on CO2 partial pressure — adequate for screening,
not for final tower rating. See `docs/thermo/amine_co2_solubility.md`.

In [ ]:
# Setup: load NeqSim from the local workspace build (devtools dev mode)
import os
import sys
from pathlib import Path


def find_neqsim_project_root():
    env_root = os.environ.get("NEQSIM_PROJECT_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root).resolve())
    cwd = Path.cwd().resolve()
    candidates.extend([cwd] + list(cwd.parents))
    for candidate in candidates:
        if (candidate / "pom.xml").exists() and (candidate / "devtools" / "neqsim_dev_setup.py").exists():
            return candidate
    raise RuntimeError("Could not find NeqSim project root. Set NEQSIM_PROJECT_ROOT.")


PROJECT_ROOT = find_neqsim_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "devtools"))

from neqsim_dev_setup import neqsim_init, neqsim_classes

ns = neqsim_init(project_root=PROJECT_ROOT, recompile=False, verbose=True)
ns = neqsim_classes(ns)
NEQSIM_MODE = "devtools"
print("NeqSim loaded via devtools (local dev mode)")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Amine classes
AmineKentEisenberg = ns.JClass("neqsim.thermo.util.amines.AmineKentEisenberg")
AmineSystem = ns.JClass("neqsim.thermo.util.amines.AmineSystem")
AmineHeatOfAbsorption = ns.JClass("neqsim.thermo.util.amines.AmineHeatOfAbsorption")

# Convenient handles to the nested enums
KE = AmineKentEisenberg
KEType = AmineKentEisenberg.AmineType  # MEA, DEA, MDEA
SysType = AmineSystem.AmineType        # MEA, DEA, MDEA, AMDEA
HoAType = AmineHeatOfAbsorption.AmineType

# Molar masses (g/mol)
MW = {"MEA": 61.08, "DEA": 105.14, "MDEA": 119.16}
T40 = 313.15  # 40 C, absorber
T100 = 373.15  # 100 C, stripper
print("Amine classes loaded successfully")

## 1. Validated screening API and regression anchors

The design-default entry point is `partialPressureCO2Bara(type, temperatureK, amineMolarity, loading)`
(static) or `AmineSystem.getCO2PartialPressure()` (wrapper). Both return the CO2
partial pressure in **bara**. Below we reproduce the values guarded by
`AmineCO2SolubilityTest`.

In [ ]:
# Solvent molarity from mass fraction and amine molar mass
mol_mdea_50 = AmineKentEisenberg.amineMolarity(0.50, MW["MDEA"])  # 50 wt% MDEA
mol_mea_30 = AmineKentEisenberg.amineMolarity(0.30, MW["MEA"])    # 30 wt% MEA

# Static model: MDEA partial pressure at 40 C
p_mdea_01 = AmineKentEisenberg.partialPressureCO2Bara(KEType.MDEA, T40, mol_mdea_50, 0.10)
p_mdea_04 = AmineKentEisenberg.partialPressureCO2Bara(KEType.MDEA, T40, mol_mdea_50, 0.40)

# Wrapper path: same MDEA case via AmineSystem
sys_mdea = AmineSystem(SysType.MDEA, T40, 1.0)
sys_mdea.setAmineConcentration(0.50)
sys_mdea.setCO2Loading(0.40)
p_mdea_04_wrapper = sys_mdea.getCO2PartialPressure()

print(f"50 wt% MDEA molarity        : {mol_mdea_50:.3f} mol/L  (expected ~4.36)")
print(f"30 wt% MEA  molarity        : {mol_mea_30:.3f} mol/L  (expected ~5.03)")
print(f"MDEA pCO2 @40C, loading 0.10: {p_mdea_01:.4f} bara  (expected ~0.0093)")
print(f"MDEA pCO2 @40C, loading 0.40: {p_mdea_04:.4f} bara  (expected ~0.229)")
print(f"MDEA pCO2 via AmineSystem    : {p_mdea_04_wrapper:.4f} bara  (matches static path)")

## 2. CO2 partial pressure vs loading — MEA vs MDEA at 40 °C

The equilibrium CO2 partial pressure over the solvent is the absorber driving
force: lower pressure at a given loading means the solvent can pick up CO2 down to
a lower gas-phase concentration. Primary MEA forms a stable carbamate and binds
CO2 tightly at low loading; tertiary MDEA forms only bicarbonate and binds CO2
more weakly, so its partial pressure is higher at the same low loading.

In [ ]:
loadings = np.linspace(0.05, 0.49, 30)

p_mea = [AmineKentEisenberg.partialPressureCO2Bara(KEType.MEA, T40, mol_mea_30, float(a)) for a in loadings]
p_mdea = [AmineKentEisenberg.partialPressureCO2Bara(KEType.MDEA, T40, mol_mdea_50, float(a)) for a in loadings]

fig, ax = plt.subplots(figsize=(7, 5))
ax.semilogy(loadings, p_mea, "o-", color="#1f77b4", label="MEA (30 wt%, primary)")
ax.semilogy(loadings, p_mdea, "s-", color="#d62728", label="MDEA (50 wt%, tertiary)")
ax.set_xlabel("CO2 loading  (mol CO2 / mol amine)")
ax.set_ylabel("CO2 partial pressure  (bara)")
ax.set_title("CO2 solubility isotherm at 40 \u00b0C — MEA vs MDEA (Kent-Eisenberg)")
ax.grid(True, which="both", ls=":", alpha=0.6)
ax.legend()
fig.tight_layout()
plt.show()

i_low = np.argmin(np.abs(loadings - 0.2))
print(f"At loading ~0.2: MEA pCO2 = {p_mea[i_low]:.4f} bara, MDEA pCO2 = {p_mdea[i_low]:.4f} bara")
print("MEA binds CO2 more tightly (lower pCO2) below half loading, as expected for a carbamate former.")

## 3. Temperature effect — absorption vs regeneration

Absorption runs cold (high CO2 capacity, low partial pressure); the stripper runs
hot to reject CO2 at high partial pressure and regenerate lean solvent. The
Henry coefficient increases strongly with temperature, so at the same loading the
hot-solvent partial pressure is several times higher than the cold-solvent value.

In [ ]:
load_mdea = np.linspace(0.05, 0.9, 30)
p_cold = [AmineKentEisenberg.partialPressureCO2Bara(KEType.MDEA, T40, mol_mdea_50, float(a)) for a in load_mdea]
p_hot = [AmineKentEisenberg.partialPressureCO2Bara(KEType.MDEA, T100, mol_mdea_50, float(a)) for a in load_mdea]

fig, ax = plt.subplots(figsize=(7, 5))
ax.semilogy(load_mdea, p_cold, "o-", color="#1f77b4", label="40 \u00b0C (absorber)")
ax.semilogy(load_mdea, p_hot, "^-", color="#ff7f0e", label="100 \u00b0C (stripper)")
ax.set_xlabel("CO2 loading  (mol CO2 / mol amine)")
ax.set_ylabel("CO2 partial pressure  (bara)")
ax.set_title("50 wt% MDEA — temperature swing drives stripping")
ax.grid(True, which="both", ls=":", alpha=0.6)
ax.legend()
fig.tight_layout()
plt.show()

i_mid = np.argmin(np.abs(load_mdea - 0.4))
ratio = p_hot[i_mid] / p_cold[i_mid]
print(f"At loading ~0.4: cold pCO2 = {p_cold[i_mid]:.4f} bara, hot pCO2 = {p_hot[i_mid]:.4f} bara")
print(f"Hot/cold partial-pressure ratio = {ratio:.1f}x  (a large ratio means easy thermal regeneration)")

## 4. Differential heat of absorption

The heat of absorption sets the reboiler duty for solvent regeneration. Primary
and secondary amines (MEA, DEA) release more heat per mole of CO2 (stronger
carbamate bond) than tertiary MDEA — a trade-off between capacity, kinetics, and
regeneration energy.

In [ ]:
alpha = np.linspace(0.05, 0.45, 25)
amines = [("MEA", HoAType.MEA, 0.30, "#1f77b4"),
          ("DEA", HoAType.DEA, 0.30, "#2ca02c"),
          ("MDEA", HoAType.MDEA, 0.50, "#d62728")]

fig, ax = plt.subplots(figsize=(7, 5))
for name, atype, conc, color in amines:
    dh = []
    for a in alpha:
        hoa = AmineHeatOfAbsorption(atype, conc, float(a), T40)
        dh.append(abs(hoa.calcHeatOfAbsorptionCO2()))
    ax.plot(alpha, dh, "o-", color=color, label=name)
    print(f"{name:4s} heat of absorption @ loading 0.3: {dh[np.argmin(np.abs(alpha - 0.3))]:.1f} kJ/mol CO2")

ax.set_xlabel("CO2 loading  (mol CO2 / mol amine)")
ax.set_ylabel("|Heat of absorption|  (kJ/mol CO2)")
ax.set_title("Differential heat of CO2 absorption at 40 \u00b0C")
ax.grid(True, ls=":", alpha=0.6)
ax.legend()
fig.tight_layout()
plt.show()

## Summary

- The validated screening API — `AmineKentEisenberg.partialPressureCO2Bara(...)`
  and `AmineSystem.getCO2PartialPressure()` — reproduces the regression anchors
  (50 wt% MDEA molarity ≈ 4.36 mol/L; MDEA pCO2 ≈ 0.0093 bara at loading 0.1 and
  ≈ 0.229 bara at loading 0.4, both at 40 °C).
- **MEA binds CO2 more tightly than MDEA below half loading** (carbamate vs
  bicarbonate), giving a lower equilibrium partial pressure at low loading.
- A large **hot/cold partial-pressure ratio** (>5×) for the same loading is what
  makes thermal regeneration of the rich solvent practical.
- Primary/secondary amines release more **heat of absorption** per mole of CO2
  than tertiary MDEA — higher binding strength at the cost of reboiler duty.

The model is intended for screening (factor-of-two accuracy on CO2 partial
pressure in the engineering loading windows). For rigorous electrolyte
equilibrium, `AmineSystem.getCO2PartialPressureRigorous()` exists but is
experimental and not yet calibrated. See `docs/thermo/amine_co2_solubility.md`.